# CATH / ProtTucker / AlphaFold2 preprocessing

Input data for Figures 5A and 5B

In [ ]:
import os
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

In [ ]:
_cwd = Path.cwd()
REPO_ROOT = next((str(p) for p in [_cwd, *_cwd.parents] if (p / ".git").exists()), str(_cwd))

FASTA_PATH       = os.path.join(REPO_ROOT, "examples/paper/data/cath/raw/test300.fasta")
CATH_LABELS_PATH = os.path.join(REPO_ROOT, "examples/paper/data/cath/raw/cath-domain-list.txt")
OUTPUT_FASTA     = os.path.join(REPO_ROOT, "examples/paper/data/cath/processed/test300.fasta")
OUTPUT_CSV       = os.path.join(REPO_ROOT, "examples/paper/data/cath/processed/test300_features.csv")

## 1. Download raw data

- **test300.fasta** — from the [EAT / ProtTucker repository](https://github.com/Rostlab/EAT) (Heinzinger et al., 2022)
- **cath-domain-list.txt** — CATH v4.3 domain list archived with the EAT dataset on [Zenodo record 14675997](https://zenodo.org/records/14675997)

Downloads are skipped if the files are already present at the paths defined above.

In [ ]:
_DOWNLOADS = {
    FASTA_PATH:       "https://raw.githubusercontent.com/Rostlab/EAT/main/data/ProtTucker/test300.fasta",
    CATH_LABELS_PATH: "https://zenodo.org/records/14675997/files/cath-domain-list.txt",
}

for dest, url in _DOWNLOADS.items():
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    if not os.path.exists(dest):
        print(f"Downloading {url} ...")
        urllib.request.urlretrieve(url, dest)
        print(f"saved to {dest}")
    else:
        print(f"Already present: {dest}")

## 2. Parse FASTA sequences

In [ ]:
sequences = {}
with open(FASTA_PATH, "r") as fh:
    current_id = None
    current_seq = []
    for line in fh:
        line = line.rstrip()
        if line.startswith(">"):
            if current_id is not None:
                sequences[current_id] = "".join(current_seq)
            current_id = line[1:].split()[0]
            current_seq = []
        else:
            current_seq.append(line)
    if current_id is not None:
        sequences[current_id] = "".join(current_seq)

print(f"Parsed {len(sequences)} sequences")

## 3. Parse CATH domain list

The CATH domain list (`cath-domain-list.txt`) is a whitespace-delimited file with columns:
`domain_id  class  architecture  topology  homologous_superfamily  s35  s60  s95  s100  s100_num  domain_length  structure_resolution`

In [ ]:
column_names = [
    "domain_id", "class", "architecture", "topology",
    "homologous_superfamily",
    "s35_sequence_cluster", "s60_sequence_cluster",
    "s95_sequence_cluster", "s100_sequence_cluster",
    "s100_sequence_cluster_number",
    "domain_length", "structure_resolution",
]

labels = pd.read_csv(
    CATH_LABELS_PATH,
    delim_whitespace=True,
    comment="#",
    header=None,
    names=column_names,
)

# Subset to the 300 test domains
labels_test300 = labels[labels["domain_id"].isin(sequences.keys())].copy()
print(f"Matched CATH entries: {len(labels_test300)}")

## 4. Feature engineering

Add `A-` prefix to CATH class (so class 1 → `A-1`), add sequence length, and bin auxiliary numeric features.

In [ ]:
labels_test300["class"]                    = "A-" + labels_test300["class"].astype(str)
labels_test300["architecture"]             = "C-" + labels_test300["architecture"].astype(str)
labels_test300["topology"]                 = "T-" + labels_test300["topology"].astype(str)
labels_test300["homologous_superfamily"]   = "H-" + labels_test300["homologous_superfamily"].astype(str)
for col in ["s35_sequence_cluster", "s60_sequence_cluster", "s95_sequence_cluster", "s100_sequence_cluster"]:
    labels_test300[col] = "seq_clus_" + labels_test300[col].astype(str)

# Sequence length from FASTA
labels_test300["sequence_length"] = labels_test300["domain_id"].map(
    lambda d: len(sequences.get(d, ""))
)

# Fixed 50-unit bins: 0-50, 50-100, ..., 350-400, 400+
bin_edges  = list(range(0, 401, 50)) + [float("inf")]
bin_labels = [f"{bin_edges[i]:.0f}-{bin_edges[i+1]:.0f}" for i in range(len(bin_edges) - 2)] + ["400+"]
labels_test300["sequence_length_bins"] = pd.cut(
    labels_test300["sequence_length"],
    bins=bin_edges,
    labels=bin_labels,
    include_lowest=True,
    right=False,
)

labels_test300.head()

## 5. Save outputs

In [ ]:
os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)
os.makedirs(os.path.dirname(OUTPUT_FASTA), exist_ok=True)

# Save features CSV
labels_test300.to_csv(OUTPUT_CSV, index=False)
print(f"Saved features to {OUTPUT_CSV}")

# Save FASTA copy (same content, standardised location)
with open(OUTPUT_FASTA, "w") as fh:
    for domain_id in labels_test300["domain_id"]:
        fh.write(f">{domain_id}\n{sequences[domain_id]}\n")
print(f"Saved FASTA to {OUTPUT_FASTA}")

## 6. Class distribution

In [ ]:
counts = labels_test300["class"].value_counts().reset_index()
counts.columns = ["class", "count"]
print(counts.to_string(index=False))

cath_names = {"A-1": "Mainly alpha", "A-2": "Mainly beta",
              "A-3": "Alpha beta", "A-4": "Few secondary structures", "A-6": "Special"}
counts["description"] = counts["class"].map(cath_names)

fig = px.bar(
    counts,
    x="class",
    y="count",
    text="count",
    hover_data=["description"],
    title="CATH test300 — class distribution",
    labels={"class": "CATH class", "count": "# domains"},
    template="plotly_white",
)
fig.show()